In [4]:
import requests
import random
from datetime import datetime, timedelta

# ==============================
# CONFIG
# ==============================

TOKEN = "your_hubspot_token_here"
SCHEMA_NAME = "purchase_order"

BASE_URL = "https://api.hubapi.com"
SCHEMA_URL = f"{BASE_URL}/crm/v3/schemas"

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json",
}


# ==============================
# 1. SCHEMA: GET OR CREATE
# ==============================

def get_or_create_purchase_order_schema() -> str:
    """
    Returns fullyQualifiedName of the purchase_order custom object,
    e.g. "p442646332_purchase_order".
    Creates schema if it does not exist.
    """

    # ---- Try to find existing schema by name ----
    r = requests.get(SCHEMA_URL, headers=HEADERS)
    r.raise_for_status()
    data = r.json()

    for s in data.get("results", []):
        if s.get("name") == SCHEMA_NAME:
            fq_name = s.get("fullyQualifiedName")
            print(f"✅ Found existing schema: {fq_name}")
            return fq_name

    # ---- Not found → create new schema ----
    schema_payload = {
        "name": SCHEMA_NAME,
        "labels": {
            "singular": "Purchase Order",
            "plural": "Purchase Orders"
        },
        "primaryDisplayProperty": "po_number",
        "requiredProperties": ["po_number"],
        "metaType": "PORTAL_SPECIFIC",
        "properties": [
            # CORE FIELDS
            {"name": "po_number", "label": "PO Number", "type": "string", "fieldType": "text"},
            {"name": "description", "label": "Description (Job Name)", "type": "string", "fieldType": "text"},
            {"name": "supplier_email", "label": "Supplier Email", "type": "string", "fieldType": "text"},
            {"name": "product_type", "label": "Product Type", "type": "string", "fieldType": "text"},
            {"name": "order_type", "label": "Order Type", "type": "string", "fieldType": "text"},
            {"name": "shipping_method", "label": "Shipping Method", "type": "string", "fieldType": "text"},
            {"name": "status", "label": "Status", "type": "string", "fieldType": "text"},

            # DATES
            {"name": "order_date", "label": "Order Date", "type": "date", "fieldType": "date"},
            {"name": "required_date", "label": "Required Date", "type": "date", "fieldType": "date"},

            # SUPPLIER PORTAL FIELDS
            {"name": "factory_process", "label": "Factory Process", "type": "string", "fieldType": "text"},
            {"name": "supplier_status", "label": "Supplier Status", "type": "string", "fieldType": "text"},
            {
                "name": "supplier_expected_despatch_date",
                "label": "Supplier Expected Despatch Date",
                "type": "date",
                "fieldType": "date"
            },

            # VALUES / MATCH
            {"name": "order_value", "label": "Order Value", "type": "number", "fieldType": "number"},
            {
                "name": "order_value_match",
                "label": "Order Value Match",
                "type": "enumeration",
                "fieldType": "select",
                "options": [
                    {"label": "Yes", "value": "yes"},
                    {"label": "No", "value": "no"}
                ]
            },
        ]
    }

    r = requests.post(SCHEMA_URL, json=schema_payload, headers=HEADERS)
    print("Schema create status:", r.status_code, r.text)
    r.raise_for_status()
    created = r.json()
    fq_name = created.get("fullyQualifiedName")
    print(f"✅ Created new schema: {fq_name}")
    return fq_name


# ==============================
# 2. DATA INSERTION
# ==============================

# Real mapped rows with real portal amounts (numeric)
REAL_PORTAL_ROWS = [
    ("50164",   "NAB BBC GREGORY HILLS 15121", 2493.26),
    ("50144",   "WARAKIRRI COLLEGE",           0.00),
    ("50145",   "WARAKIRRI COLLEGE",           0.00),
    ("50142",   "IRT ST GEORGE BASIN 15125",   2589.95),
    ("50135",   "ESTIA GYMPIE 13520",          765.05),
    ("41400",   "OXFORD ST RES 12929",         1103.96),
    ("50075",   "UNITING AGE CARE GRIF 15112", 1851.30),
    ("43420",   "SYDNEY CHILDRENS WEST 13439", 529.32),
    ("43345",   "SCAPE ASCOT 11064",           2197.80),
    ("42333",   "BOLTON CLARKE BURL 13021",    3884.81),
    ("41964",   "BUILDCORP L4 13265",          1133.83),
    ("PO41301", "PAUL LEE RESIDENCE",          238.70),
    ("40375",   "RAN609263",                   0.00),
    ("39914",   "ABF NEUTRAL BAY 12518",       11683.10),
]

PRODUCTS = ["Fabric", "Roller Blinds", "Curtain Tracks", "Shutters"]
ORDER_TYPES = ["Standard", "Defect Order", "Product Return"]
SHIPPING = ["Direct NSW", "Direct QLD", "Direct to Workroom"]


def build_properties(po_number: str, job: str, order_value: float) -> dict:
    order_date = datetime.now() - timedelta(days=random.randint(1, 10))
    required_date = order_date + timedelta(days=random.randint(10, 30))
    status = "Order Issued"

    return {
        "po_number": po_number,
        "description": job,
        "supplier_email": "supplier_test@portal.com",
        "product_type": random.choice(PRODUCTS),
        "order_type": random.choice(ORDER_TYPES),
        "shipping_method": random.choice(SHIPPING),
        "status": status,

        # Dates as "YYYY-MM-DD" (valid for type=date)
        "order_date": order_date.strftime("%Y-%m-%d"),
        "required_date": required_date.strftime("%Y-%m-%d"),

        # Numeric value
        "order_value": float(order_value),

      
    }


def insert_record(object_type: str, props: dict):
    url = f"{BASE_URL}/crm/v3/objects/{object_type}"
    r = requests.post(url, json={"properties": props}, headers=HEADERS)
    print(f"Insert {props['po_number']} → {r.status_code}")
    if r.status_code >= 400:
        print("  Body:", r.text)


def insert_sample_data(object_type: str):
    # 1) Real portal rows
    for po, job, amt in REAL_PORTAL_ROWS:
        props = build_properties(po, job, amt)
        insert_record(object_type, props)

    # 2) Noise/test rows
    for i in range(80):
        po = str(7000000 + i)
        job = f"TEST PROJECT {i}"
        # random amount like 1234.56
        amt = float(f"{random.randint(50, 12000)}.{random.randint(0, 99):02d}")
        props = build_properties(po, job, amt)
        insert_record(object_type, props)


# ==============================
# MAIN
# ==============================

if __name__ == "__main__":
    fq_object_type = get_or_create_purchase_order_schema()
    insert_sample_data(fq_object_type)


✅ Found existing schema: p442646332_purchase_order
Insert 50164 → 201
Insert 50144 → 201
Insert 50145 → 201
Insert 50142 → 201
Insert 50135 → 201
Insert 41400 → 201
Insert 50075 → 201
Insert 43420 → 201
Insert 43345 → 201
Insert 42333 → 201
Insert 41964 → 201
Insert PO41301 → 201
Insert 40375 → 201
Insert 39914 → 201
Insert 7000000 → 201
Insert 7000001 → 201
Insert 7000002 → 201
Insert 7000003 → 201
Insert 7000004 → 201
Insert 7000005 → 201
Insert 7000006 → 201
Insert 7000007 → 201
Insert 7000008 → 201
Insert 7000009 → 201
Insert 7000010 → 201
Insert 7000011 → 201
Insert 7000012 → 201
Insert 7000013 → 201
Insert 7000014 → 201
Insert 7000015 → 201
Insert 7000016 → 201
Insert 7000017 → 201
Insert 7000018 → 201
Insert 7000019 → 201
Insert 7000020 → 201
Insert 7000021 → 201
Insert 7000022 → 201
Insert 7000023 → 201
Insert 7000024 → 201
Insert 7000025 → 201
Insert 7000026 → 201
Insert 7000027 → 201
Insert 7000028 → 201
Insert 7000029 → 201
Insert 7000030 → 201
Insert 7000031 → 201
Insert 70